# E1 — ClinVar and gnomAD

**The plan for this program was fixed before it was written.**
[`docs/preregistration/E1.md`](../docs/preregistration/E1.md), committed at `07d2b81` with an
amendment at `e9b509e`. Part 6 governs. Nothing below deviates from it; anything not specified
there is exploratory and labelled as such.

Two questions:

1. **Q1** — when the model says a variant breaks the protein, do clinical databases call it
   disease-causing?
2. **Q2** — do healthy people carry fewer protein-breaking variants than chance predicts?

### The predictions were fixed before the labels were attached

Scores come from **B6** (22 Sep 2026), which scored all 366,168 possible substitutions in the
310 SH3 domains for an unrelated question. They pre-date this pre-registration by five days and
were produced without reference to ClinVar or gnomAD, so they cannot have been influenced by the
labels even in principle. No inference runs here — this program is a join and a set of tests.

### Arms (Part 6.1)

| Arm | ClinVar stars | "Common" | Verdict? |
|---|---|---|---|
| **1 — MAIN** | 1+ | popmax > 1 % | **yes** |
| 3 | 0+ | popmax > 1 % | yes |
| 4 | 1+ | popmax > 5 % | yes |
| 2 | 2+ | popmax > 1 % | no — 4 variants |
| 6 | atlas only | popmax > 1 % | no — 3 variants |

Arms 2 and 6 run and report counts; their sizes are the finding. Arm 5 was dropped, with the
reason recorded in Part 6.1.

### Power, stated in advance (Part 6.3)

**21 disease-causing against ~120 harmless detects a large effect and nothing smaller.**
(126 by domain position; a few drop where ClinVar''s transcript disagrees with UniProt.) A null
in Q1 is therefore ambiguous between "the calls do not track pathogenicity" and "this test
could not have seen it", and is reported that way. The **continuous** comparison is primary; the
rate comparison is secondary. Effect sizes with intervals are reported whether or not a verdict
is possible.

Significance: **p < 0.025** (0.05 across two questions). Fewer than **20** in a group → no verdict.

Runtime: CPU, about two minutes.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time
from pathlib import Path
import pandas as pd, numpy as np
from scipy.stats import mannwhitneyu, fisher_exact, binomtest

DRIVE = Path('/content/drive/MyDrive')

def pick(*names, what=''):
    for n in names:
        if (DRIVE / n).exists(): return DRIVE / n
    raise FileNotFoundError(f'No {what} folder in MyDrive. Tried: ' + ', '.join(names))

PROJECT = pick('sh3_repro/extended', 'SH3_extended', 'SH3 extended', what='project')
FROZEN  = pick('sh3_repro', what='frozen-run')          # READ ONLY
FROZEN_IN = PROJECT / 'input' / 'frozen'
XOUT    = PROJECT / 'results' / 'X0_external_data'
OUT     = PROJECT / 'results' / 'E1_clinvar_gnomad'
OUT.mkdir(parents=True, exist_ok=True)

def frozen(rel, bundled):
    p = FROZEN / 'results' / rel
    return p if p.exists() else FROZEN_IN / 'results' / bundled

print('project:', PROJECT); print('frozen :', FROZEN)

In [ ]:
# ── pinned inputs, per Part 6 ─────────────────────────────────────────────
ALPHA, MIN_N = 0.025, 20

V = pd.read_parquet(frozen('B6_virtual_saturation_mutagenesis/sh3_virtual_dms.parquet',
                           'sh3_virtual_dms.parquet'))
print(f'B6 virtual scan: {len(V):,} substitutions, {V.domain_key.nunique()} domains')
assert len(V) == 366168, 'B6 table is not the pinned one'

cv = pd.read_csv(XOUT / 'clinvar_sh3.csv')
gn = pd.read_csv(XOUT / 'gnomad_sh3_missense.csv')

# X0 passed ClinVar's own header through unchanged; the atlas and gnomAD both say 'gene'
cv = cv.rename(columns={'GeneSymbol': 'gene'})
for _name, _d in (('ClinVar', cv), ('gnomAD', gn)):
    _miss = {'gene', 'pos_protein', 'wt_aa', 'mut_aa'} - set(_d.columns)
    assert not _miss, f'{_name} is missing {_miss}; X0 output columns changed'

if 'popmax' not in gn.columns:
    raise SystemExit('gnomad_sh3_missense.csv has no popmax column. Re-run the gnomAD '
                     'cell of X0; its query was extended per Part 6.4 of the plan.')
print(f'ClinVar {len(cv):,} records | gnomAD {len(gn):,} variants')

## 1. Restrict to SH3 domain boundaries and attach the scores

B6 is keyed by `domain_key` and a **domain-local** position; ClinVar and gnomAD are keyed by gene
and **absolute protein** position. The atlas supplies the boundaries that convert between them.

This is the same class of trap as E2's Rocklin mapping — joining on the wrong position system
returns zero rows silently.

In [ ]:
atlas = pd.read_csv(frozen('A0_cancer_atlas/SH3_cancer_master.tsv', 'SH3_cancer_master.tsv'),
                    sep='\t', usecols=['gene','uniprot','domain_idx','dom_start','dom_end',
                                       'pos_protein','wt_aa','mut_aa','pos_in_domain'])
# the atlas has no domain_key column; B6 keys on uniprot + domain index
atlas['domain_key'] = atlas.uniprot + '_d' + atlas.domain_idx.astype(int).astype(str)
assert (atlas.pos_protein - atlas.dom_start + 1 == atlas.pos_in_domain).all(), \
       'dom_start convention differs from the atlas pos_in_domain'
dom = atlas[['domain_key','gene','uniprot','domain_idx','dom_start','dom_end']].drop_duplicates()
print(f'{len(dom)} domains define the boundaries')

# the scores E1 uses are the same frozen head B3 published: verified here, not assumed
b3 = pd.read_parquet(frozen('B3_uncertainty_and_calls/atlas_predictions.parquet',
                            'atlas_predictions.parquet'),
                     columns=['domain_key','pos','wt_aa','mut_aa','pred_ddG_fold'])
_c = b3.merge(V[['domain_key','pos','wt_aa','mut_aa','pred_ddG_fold']],
              on=['domain_key','pos','wt_aa','mut_aa'], suffixes=('_b3','_b6'))
_d = (_c.pred_ddG_fold_b3 - _c.pred_ddG_fold_b6).abs().max()
assert len(_c) == len(b3) and _d < 1e-4, f'B6 scores differ from B3 ({_d:.2e})'
print(f'B6 reproduces all {len(_c):,} B3 predictions, max diff {_d:.1e}')

DROPPED = {}

def attach(df, label):
    """gene + absolute position -> domain_key + domain-local position, then B6 scores."""
    j = df.merge(dom, on='gene')
    j = j[(j.pos_protein >= j.dom_start) & (j.pos_protein <= j.dom_end)].copy()
    j['pos'] = j.pos_protein - j.dom_start + 1
    out = j.merge(V[['domain_key','pos','wt_aa','mut_aa','pred_ddG_fold','sd_fold',
                     'fold_lo','destabilising','snv_reachable','ref_pos']],
                  on=['domain_key','pos','wt_aa','mut_aa'], how='inner')
    DROPPED[label] = (len(j), len(out))
    return out

CV, GN = attach(cv, 'ClinVar'), attach(gn, 'gnomAD')
print(f'inside domains and scored:  ClinVar {len(CV):,}   gnomAD {len(GN):,}')
# the gap is not a bug: a substitution can only be scored where its wild-type residue exists
# in the UniProt canonical sequence B6 modelled, and ClinVar picks its own transcript.
for _k, (_a, _b) in DROPPED.items():
    print(f'  {_k}: {_a:,} inside domain bounds, {_b:,} scored '
          f'({_a - _b:,} wild-type residue disagrees with the canonical sequence)')
assert len(CV) and len(GN), 'position mapping produced no rows - check dom_start convention'

In [ ]:
# ── label groups, per Part 1 of the pre-registration ─────────────────────
sig = CV.ClinicalSignificance.str.lower()
CV['group'] = np.where(sig.str.match(r'^(likely )?pathogenic$'), 'pathogenic',
              np.where(sig.str.match(r'^(likely )?benign$'),     'benign', None))
CV = CV[CV.group.notna()].drop_duplicates(['gene','pos_protein','wt_aa','mut_aa','group','stars'])

ATLAS_KEYS = set(zip(atlas.gene, atlas.pos_protein, atlas.wt_aa, atlas.mut_aa))
CV['in_atlas'] = [(g,p,w,m) in ATLAS_KEYS for g,p,w,m in
                  zip(CV.gene, CV.pos_protein, CV.wt_aa, CV.mut_aa)]
print(CV.groupby(['group', CV.stars.clip(upper=3)]).size().unstack(fill_value=0).to_string())

## 2. Q1 — do the calls track pathogenicity?

Continuous first (Part 6.3): the predicted ΔΔG distributions, by Mann–Whitney with a rank-biserial
effect size. The rate comparison follows as the secondary presentation.

In [ ]:
def rank_biserial(a, b):
    u = mannwhitneyu(a, b, alternative='two-sided').statistic
    return 2*u/(len(a)*len(b)) - 1        # +1 = a entirely above b

def q1(df, stars, label):
    d = df[df.stars >= stars]
    p, b = d[d.group=='pathogenic'].pred_ddG_fold, d[d.group=='benign'].pred_ddG_fold
    n_p, n_b = len(p), len(b)
    row = dict(arm=label, n_pathogenic=n_p, n_benign=n_b)
    if n_p < MIN_N or n_b < MIN_N:
        return {**row, 'verdict': 'too few', 'p': np.nan, 'effect': np.nan,
                'median_path': p.median() if n_p else np.nan,
                'median_benign': b.median() if n_b else np.nan}
    stat, pv = mannwhitneyu(p, b, alternative='greater')   # destabilising = higher ddG
    return {**row, 'verdict': 'yes' if pv < ALPHA else 'no', 'p': pv,
            'effect': rank_biserial(p, b),
            'median_path': p.median(), 'median_benign': b.median()}

Q1 = pd.DataFrame([q1(CV, 1, '1 - MAIN (1+ stars)'), q1(CV, 0, '3 (0+ stars)'),
                   q1(CV, 1, '4 (1+ stars, AF>5%)'), q1(CV, 2, '2 (2+ stars)'),
                   q1(CV[CV.in_atlas], 1, '6 (atlas only)')])
print(Q1.to_string(index=False))

In [ ]:
# secondary: the rate comparison
def q1_rate(df, stars, label):
    d = df[df.stars >= stars]
    p, b = d[d.group=='pathogenic'], d[d.group=='benign']
    if len(p) < MIN_N or len(b) < MIN_N:
        return dict(arm=label, verdict='too few', rate_path=p.destabilising.mean() if len(p) else np.nan,
                    rate_benign=b.destabilising.mean() if len(b) else np.nan, p=np.nan)
    tab = [[int(p.destabilising.sum()), int((~p.destabilising).sum())],
           [int(b.destabilising.sum()), int((~b.destabilising).sum())]]
    _, pv = fisher_exact(tab, alternative='greater')
    return dict(arm=label, verdict='yes' if pv < ALPHA else 'no',
                rate_path=p.destabilising.mean(), rate_benign=b.destabilising.mean(), p=pv)

Q1R = pd.DataFrame([q1_rate(CV, 1, '1 - MAIN'), q1_rate(CV, 0, '3'), q1_rate(CV, 2, '2'),
                    q1_rate(CV[CV.in_atlas], 1, '6')])
print(Q1R.to_string(index=False))
print(f'\nBeltran et al. report 60% of pathogenic missense variants reduce stability.')

## 3. Q2 — are common variants depleted of destabilising substitutions?

"Chance" is B6's null, exactly as in the published comparison (5.9 % observed against 9.2 %
expected in cancer): the single-nucleotide-reachable substitutions available at the same
positions. Common variants must not be labelled pathogenic in ClinVar.

In [ ]:
PATH_KEYS = set(zip(CV[CV.group=='pathogenic'].gene, CV[CV.group=='pathogenic'].pos_protein,
                    CV[CV.group=='pathogenic'].wt_aa, CV[CV.group=='pathogenic'].mut_aa))
GN['is_pathogenic'] = [(g,p,w,m) in PATH_KEYS for g,p,w,m in
                       zip(GN.gene, GN.pos_protein, GN.wt_aa, GN.mut_aa)]

def q2(cut, label, restrict_atlas=False):
    g = GN[(GN.popmax > cut) & (~GN.is_pathogenic)]
    if restrict_atlas:
        g = g[[(a,p,w,m) in ATLAS_KEYS for a,p,w,m in zip(g.gene,g.pos_protein,g.wt_aa,g.mut_aa)]]
    row = dict(arm=label, n_common=len(g), observed=g.destabilising.mean() if len(g) else np.nan)
    doms = set(g.domain_key)
    null = V[V.domain_key.isin(doms) & V.snv_reachable]
    row['expected'] = null.destabilising.mean() if len(null) else np.nan
    if len(g) < MIN_N:
        return {**row, 'verdict': 'too few', 'p': np.nan}
    k, n, q = int(g.destabilising.sum()), len(g), row['expected']
    pv = binomtest(k, n, q, alternative='less').pvalue
    return {**row, 'verdict': 'yes' if pv < ALPHA else 'no', 'p': pv}

Q2 = pd.DataFrame([q2(0.01, '1 - MAIN (popmax>1%)'), q2(0.01, '3 (popmax>1%)'),
                   q2(0.05, '4 (popmax>5%)'), q2(0.01, '2 (popmax>1%)'),
                   q2(0.01, '6 (atlas only)', restrict_atlas=True)])
print(Q2.to_string(index=False))
print(f'\nB6 reported 5.9% observed against 9.2% expected for cancer variants.')

## 4. The comparison table, read by the rule fixed in advance

Part 3 of the pre-registration: the verdict is arm 1's, always. Agreement across arms means the
result does not depend on our choices; disagreement is named as a finding, not hidden.

In [ ]:
SUM = (Q1[['arm','n_pathogenic','n_benign','verdict','p','effect']]
         .rename(columns={'verdict':'Q1','p':'Q1_p','effect':'Q1_effect'})
         .merge(Q2[['arm','n_common','verdict','observed','expected','p']]
                  .rename(columns={'verdict':'Q2','p':'Q2_p'}), on='arm', how='outer'))
SUM.to_csv(OUT / 'summary_E1.csv', index=False)
print(SUM.to_string(index=False))

main = Q1[Q1.arm.str.startswith('1 -')].iloc[0]
# the AF threshold changes nothing in Q1, so arm 4 is the SAME test as arm 1; counting it as
# corroboration would be circular. Symmetrically the star threshold changes nothing in Q2.
others = Q1[~Q1.arm.str.startswith('1 -') & (Q1.verdict != 'too few')]
dup = others.apply(lambda r: (r.n_pathogenic, r.n_benign) ==
                             (main.n_pathogenic, main.n_benign), axis=1)
if len(others):
    print('arms identical to the main test, so not corroboration:',
          list(others.arm[dup]) or 'none')
    others = others[~dup]
robust = bool(len(others)) and bool((others.verdict == main.verdict).all())
print(f'\nMAIN verdict (Q1): {main.verdict}')
print('robust across arms with enough data' if robust else
      'arms disagree - name which choice flips it' if len(others) else
      'no other arm had enough data to corroborate')

## 5. RESULTS_E1.md and metrics_E1.json

In [ ]:
metrics = {'program':'E1', 'date': time.strftime('%Y-%m-%d %H:%M'),
 'preregistration':'docs/preregistration/E1.md @ 07d2b81, amended e9b509e',
 'alpha': ALPHA, 'min_group_n': MIN_N,
 'clinvar_in_domains': int(len(CV)), 'gnomad_in_domains': int(len(GN)),
 'main_Q1_verdict': main.verdict, 'main_Q1_p': None if pd.isna(main.p) else float(main.p),
 'main_Q1_effect': None if pd.isna(main.effect) else float(main.effect),
 'robust_across_arms': robust,
 'Q1': Q1.to_dict('records'), 'Q1_rate': Q1R.to_dict('records'), 'Q2': Q2.to_dict('records')}
(OUT / 'metrics_E1.json').write_text(json.dumps(metrics, indent=1, default=str))

L = ['# E1 - ClinVar and gnomAD', '',
     f"Run {metrics['date']}. Machine-written. Plan fixed in advance: "
     '`docs/preregistration/E1.md` @ 07d2b81, amended e9b509e.', '',
     'Predictions come from B6 (22 Sep 2026), computed for an unrelated question five days '
     'before this plan and without reference to any clinical label.', '',
     f'ClinVar variants inside SH3 domains: {len(CV):,}. gnomAD: {len(GN):,}.', '',
     '## Q1 - do the calls track pathogenicity? (continuous, primary)', '',
     '| arm | n path | n benign | verdict | p | effect |', '|---|---|---|---|---|---|']
L += [f"| {r.arm} | {r.n_pathogenic} | {r.n_benign} | {r.verdict} | "
      f"{'-' if pd.isna(r.p) else format(r.p, '.3g')} | "
      f"{'-' if pd.isna(r.effect) else format(r.effect, '+.3f')} |" for r in Q1.itertuples()]
L += ['', '## Q2 - are common variants depleted? (B6 null)', '',
      '| arm | n common | observed | expected | verdict | p |', '|---|---|---|---|---|---|']
L += [f"| {r.arm} | {r.n_common} | "
      f"{'-' if pd.isna(r.observed) else format(r.observed, '.1%')} | "
      f"{'-' if pd.isna(r.expected) else format(r.expected, '.1%')} | {r.verdict} | "
      f"{'-' if pd.isna(r.p) else format(r.p, '.3g')} |" for r in Q2.itertuples()]
L += ['', '## Reading, by the rule fixed in advance', '',
      f'The verdict is arm 1\'s: **{main.verdict}**.',
      ('Every arm with enough data agrees, so the result does not depend on these choices.'
       if robust else 'Arms disagree; the differing choice is named in ANALYSIS_E1.md.'
       if len(others) else 'No other arm had enough data to corroborate.'), '',
      f'**Power.** {main.n_pathogenic} disease-causing against {main.n_benign} harmless '
      'detects a large effect and nothing smaller; a null here is ambiguous between absence '
      'and insufficient power, as stated in the pre-registration before any result existed. '
      f'The pre-registration counted 21 / 126 by domain position alone; {main.n_benign} '
      'benign also have their wild-type residue in the sequence B6 scored.']
(OUT / 'RESULTS_E1.md').write_text('\n'.join(L))
print('\n'.join(L))